# 例子和测试: 山地车环境使用 sac 算法训练智能体

**关键点: 重写合适的reward**

## import

In [ ]:
import matplotlib.pyplot as plt
from torch import Tensor, concat, manual_seed
from torch.nn import Linear, ReLU, Sequential
from torch.nn.functional import softplus
from torch.nn.init import xavier_uniform_, zeros_

from modelsolver import AgentModelSolver
from modelsolver.abc.config import (AgentConfig, HyperParameterConfig,
                                    )
from modelsolver.abc.model import IActor, ICritic
from modelsolver.implement.environment.mountaincar import MountainCarEnvironment, MountainCarReward
from modelsolver.implement.data.hereplaybuffer import SimpleHEReplay, HEReplayConfig
manual_seed(0)  # 设置随机种子以确保结果可复现

## 定义智能体模型

In [ ]:
class SimpleActor(IActor):
    def __init__(self, config: AgentConfig):
        super().__init__()
        self._config = config
        self.sequential = self._create_policy_network()
        self.mean = Linear(self.config.hidden_channels, self.config.action_channels)
        self.std = Linear(self.config.hidden_channels, self.config.action_channels)
        self._init_params()

    @property
    def config(self) -> AgentConfig:
        return self._config  # type: ignoree

    def _forward(self, state: Tensor) -> Tensor:
        return self.sequential(state)

    def _action_mean(self, x: Tensor) -> Tensor:
        return self.mean(x)

    def _action_std(self, x: Tensor) -> Tensor:
        return softplus(self.std(x))

    def _create_policy_network(self):

        return Sequential(
            Linear(self.config.state_channels, self.config.hidden_channels),
            ReLU(),
            Linear(self.config.hidden_channels, self.config.hidden_channels),
            ReLU(),
            Linear(self.config.hidden_channels, self.config.hidden_channels),
            ReLU(),
        )

    def _init_params(self):
        for module in self.children():
            if isinstance(module, Linear):
                xavier_uniform_(module.weight)
                zeros_(module.bias)


class SimpleCritic(ICritic):
    def __init__(self, config: AgentConfig):
        super().__init__()
        self._config = config
        self.sequential = self._create_value_network()
        self._init_params()

    @property
    def config(self) -> AgentConfig:
        return self._config  # type: ignore

    def forward(self, states, actions):
        return self.sequential(concat([states, actions], dim=-1))

    def _create_value_network(self):

        return Sequential(
            Linear(self.config.state_channels + self.config.action_channels, self.config.hidden_channels),
            ReLU(),
            Linear(self.config.hidden_channels, self.config.hidden_channels),
            ReLU(),
            Linear(self.config.hidden_channels, self.config.hidden_channels),
            ReLU(),
            Linear(self.config.hidden_channels, 1)
        )

    def _init_params(self):
        for module in self.children():
            if isinstance(module, Linear):
                xavier_uniform_(module.weight)
                zeros_(module.bias)

## 设置配置项

In [ ]:
S = 6
A = 1
agent_config = AgentConfig(
    state_channels=S,
    action_channels=A,
    hidden_channels=512,
    target_entropy=-A,
    alpha_learnable=False
)
# sac: c:3e-2, a:3e-2 gamma_tl:0.98
hyper_params_config = HyperParameterConfig(learning_rate=5e-4,
                                           milestones=[999],
                                           gamma_rl=0.99,
                                           critic_lr=1e-3,
                                           actor_lr=5e-4,
                                           epoch=300,
                                           weight_decay=0.00,
                                        policy_delay=2)

replay_buffer_config = HEReplayConfig(
    capacity=10_0000,
    state_dim=S,
    action_dim=A,
    minimal_capacity=2048,
    batch_size=1024,
    her_threshold=0.2,
)

## 构建解决方案

In [ ]:
solver = AgentModelSolver()\
    .add_actor_component(SimpleActor)\
    .add_critic_component(SimpleCritic)\
    .add_model_config(agent_config)\
    .add_replay_buffer(SimpleHEReplay)\
    .add_environment(MountainCarEnvironment)\
    .add_reward(MountainCarReward)\
    .add_config(hyper_params_config)\
    .add_replay_buffer_config(replay_buffer_config)

## 训练过程

In [ ]:
solver.replay_buffer_warming_up(1500, "random")\
    .train(1,"sac")

使用了 HER 算法的训练:

*由于没有设计HER内的Reward和Distance方法, 效果一般, 但不至于破坏训练*

![img](./assets/mountaincar.her.gif)